# Z3 的数据类型：表达式、类型与函数声明这三者之间是什么关系

## 这一章讲什么

Z3 的那些对象，实际上被整齐地组织成了 Python 的类。把这层关系弄清楚以后，查阅 API 时就不必逐个去试，因此这一章先把继承结构弄清楚。

- `AstRef` 位于最上层，其下分出 `SortRef`(类型)、`FuncDeclRef`(函数声明)、`ExprRef`(表达式)三支
- 变量与表达式都是 `ExprRef`，`x + y` 和 `x > y` 属于同一类对象
- `sort()` 取出类型，`decl()` 取出函数声明
- 从模型(`ModelRef`)中取值时，返回的类型还会跟着变化

前面几章里随手就用的 `Int`、`Bool`、`model[x]`，这一回要把内部的结构看清楚。读者可以对照代码逐格执行，把输出中的类型名与本章的说明对应起来。

In [1]:
import z3
from z3 import *

## Z3 的类关系

下面先看全貌，这是 Z3 的类继承树，图中每一层缩进表示一层继承关系，越靠上的类越抽象。

```
└──Z3PPObject
   ├──AstRef
   │  ├──SortRef
   │  │  ├──BoolSortRef
   │  │  ├──ArithSortRef
   │  │  ├──...
   │  │
   │  ├──FuncDeclRef
   │  └──ExprRef
   │     ├──BoolRef
   │     │  └──QuantifierRef
   │     ├──ArithRef
   │     │  ├──IntNumRef
   │     │  ├──...
   │     ├──...
   ├──ModelRef
   ├──Solver
   └──...
```

`AstRef`(抽象语法树的节点)之下有三个子类，分别表示类型、函数声明和表达式，三者在程序中的分工不同，对应用法列在下表。

| 类 | 表示什么 | 例子 |
|---|---|---|
| `SortRef` | 类型(排序) | `Int`, `Bool`, `BitVec(8)` |
| `FuncDeclRef` | 函数或运算符的声明 | `x`(变量)、`>`(比较运算符) |
| `ExprRef` | 表达式 | `x`, `x + y`, `x > y` |

`Int('x')` 与 `x > y` 都是 `ExprRef` 的实例，这一点只看文字不好体会，执行下面的代码观察真实的类型就比较清楚了。

In [2]:
x, y = Ints('x y')
expr = x > y

print(f'type(x) = {type(x).__name__}')
print(f'type(expr) = {type(expr).__name__}')

type(x) = ArithRef
type(expr) = BoolRef


`x` 的类型为 `ArithRef`(算术表达式)，`expr` 的类型为 `BoolRef`(真假的表达式)，两者都是 `ExprRef` 的子类。

## ExprRef：表达式的内部结构

表达式是一棵树，既然是树就有子节点，因此可以从外层逐层向下查看。下面列出查看节点结构的两个方法。

- `num_args()` … 有几个参数
- `arg(i)` … 第 i 个参数

In [3]:
print(f'{x.num_args() = }')
print(f'{expr.num_args() = }')

x.num_args() = 0
expr.num_args() = 2


变量 `x` 不带任何参数，因此参数个数为 0；`x > y` 把 `x` 与 `y` 都当作参数，其参数个数为 2，正好对应表达式树上的两个叶子节点。

In [4]:
print(f'{expr.arg(0) = }')
print(f'{expr.arg(1) = }')

expr.arg(0) = x
expr.arg(1) = y


`expr = x > y` 的两个参数确实就是 `x` 与 `y`，可见表达式可以逐层拆开，其内部存放的内容能够完整看清，这正是 `ExprRef` 这类对象的作用。

## SortRef：表达式的类型

调用 `sort()` 返回的是表达式的类型，也就是 `SortRef` 对象，它记录的是表达式取值所属的类别，与具体的取值无关，这一点在后面的类型判断中会经常用到。

In [5]:
x_sort = x.sort()
expr_sort = expr.sort()

print(f'{x_sort = }')
print(f'{expr_sort = }')
print(f'type(x_sort) = {type(x_sort).__name__}')
print(f'type(expr_sort) = {type(expr_sort).__name__}')

x_sort = Int
expr_sort = Bool
type(x_sort) = ArithSortRef
type(expr_sort) = BoolSortRef


`x` 是整数，其类型为 `Int`，而这个类型对象所属的类是 `ArithSortRef`；`expr = x > y` 的结果是真是假，其类型为 `Bool`，所属的类是 `BoolSortRef`。

## FuncDeclRef：函数声明

调用 `decl()` 返回的是表达式的函数声明，即 `FuncDeclRef` 对象，其中包含着函数名、参数个数与各参数类型等信息，这些信息在下面的表中全部列出。在 Z3 内部，变量也好，运算符也好，一律按函数对待，因此变量与运算符在这种视角下并没有本质区别，这一点需要注意。

In [6]:
x_decl = x.decl()
expr_decl = expr.decl()

print(f'{x_decl = }')
print(f'{expr_decl = }')
print(f'type(x_decl) = {type(x_decl).__name__}')
print(f'type(expr_decl) = {type(expr_decl).__name__}')

x_decl = x
expr_decl = >
type(x_decl) = FuncDeclRef
type(expr_decl) = FuncDeclRef


- `x` 是一个不带参数的函数 `x()`，因此其声明是 `FuncDeclRef`
- `expr = x > y` 中的 `>` 也是函数 `>`，其声明同样是 `FuncDeclRef`

手上有了 `FuncDeclRef`，函数的各项信息都可以问出来，下面列出常用的几种方法。

| 方法 | 返回值 |
|---|---|
| `name()` | 函数的名字 |
| `arity()` | 参数个数 |
| `range()` | 返回值的类型(`SortRef`) |
| `domain(i)` | 第 i 个参数的类型(`SortRef`) |
| `kind()` | Z3 内部给它的编号(整数 ID) |
| `sexpr()` | S 表达式(Lisp 风格的文本) |

In [7]:
print(f'{x_decl.name() = }')
print(f'{x_decl.arity() = }')
print(f'{x_decl.range() = }')

print(f'{expr_decl.name() = }')
print(f'{expr_decl.arity() = }')
print(f'{expr_decl.range() = }')
print(f'{expr_decl.domain(0) = }')
print(f'{expr_decl.domain(1) = }')

x_decl.name() = 'x'
x_decl.arity() = 0
x_decl.range() = Int
expr_decl.name() = '>'
expr_decl.arity() = 2
expr_decl.range() = Bool
expr_decl.domain(0) = Int
expr_decl.domain(1) = Int


变量 `x` 的名字为 `x`，参数个数为 0，返回值类型是 `Int`。`>` 的名字为 `>`，参数个数为 2，返回值类型是 `Bool`，其两个参数都是 `Int`。

`kind()` 返回的是 Z3 内部的标识符，其值为一个整数，把它与 `z3` 模块中的常量对照一下，这个符号的含义就清楚了。

In [8]:
print(f'{x_decl.kind() = }   # 用户定义的名称')
print(f'{expr_decl.kind() = }   # 比较运算符 >')

print(f'{z3.Z3_OP_UNINTERPRETED = }')
print(f'{z3.Z3_OP_GT = }')

x_decl.kind() = 49167   # 用户定义的名称
expr_decl.kind() = 517   # 比较运算符 >
z3.Z3_OP_UNINTERPRETED = 49167
z3.Z3_OP_GT = 517


`x_decl.kind()` 与 `z3.Z3_OP_UNINTERPRETED` 是同一个值，说明 `x` 是用户自己定义、没有被解释的符号；`expr_decl.kind()` 则是 `z3.Z3_OP_GT`，即大于运算符，两者的差别正好体现了变量与运算符的不同。

想知道 Z3 内部持有的那条声明原样是什么，用 `sexpr()` 把它打印出来即可。

In [9]:
print(f'{x_decl.sexpr() = }')
print(f'{expr_decl.sexpr() = }')

x_decl.sexpr() = '(declare-fun x () Int)'
expr_decl.sexpr() = '(declare-fun > (Int Int) Bool)'


读出来的内容是：`x` 声明成 `(declare-fun x () Int)`，`>` 声明成 `(declare-fun > (Int Int) Bool)`，这正是 S 表达式形式的声明原文。

## 把关系画成图

```{mermaid}
graph
x["Int('x')\nArithRef"]
y["Int('y')\nArithRef"]
expr["x > y\nBoolRef"]
x_decl["x 的声明\nFuncDeclRef"]
expr_decl["> 的声明\nFuncDeclRef"]
int_sort["Int\nArithSortRef"]
bool_sort["Bool\nBoolSortRef"]

x --sort()--> int_sort
y --sort()--> int_sort
expr --sort()--> bool_sort
x --decl()--> x_decl
expr --decl()--> expr_decl
expr --arg(0)--> x
expr --arg(1)--> y
x_decl --range()--> int_sort
expr_decl --range()--> bool_sort
expr_decl --domain(0)--> int_sort
expr_decl --domain(1)--> int_sort
```

## ModelRef：从模型里取值

从 `Solver.model()` 得到的是一个 `ModelRef` 对象，这个 `ModelRef` 的 `[]` 运算与普通容器不同。传入的内容不同，得到的结果也就不同，下面分别说明这两种用法。

In [10]:
x, y, z = Ints('x y z')
solver = Solver()
solver.add(x > y, y > z, x > 0, z < 5)
solver.check()
m = solver.model()

print(f'type(m) = {type(m).__name__}')
print(f'{m[x] = }  type = {type(m[x]).__name__}')
print(f'{m[0] = }  type = {type(m[0]).__name__}')

type(m) = ModelRef
m[x] = 1  type = IntNumRef
m[0] = z  type = FuncDeclRef


- `m[x]`，传入变量 … 得到这个变量的值，类型为 `IntNumRef`
- `m[0]`，传入整数 … 得到第 0 个变量的声明，类型为 `FuncDeclRef`

`IntNumRef` 虽然是 Z3 的整数值，但它并不是 Python 的 `int`，要拿去做计算，需要先用 `as_long()` 转换一次才行。

In [11]:
print(f'{m[x].as_long() = }')
print(f'type(m[x].as_long()) = {type(m[x].as_long()).__name__}')

# 传入表达式时，返回该表达式的值（BoolRef）
print(f'{m.eval(x > y) = }  type = {type(m.eval(x > y)).__name__}')

m[x].as_long() = 1
type(m[x].as_long()) = int
m.eval(x > y) = True  type = BoolRef


经过 `as_long()` 转换以后，得到的才是 Python 的整数，可以直接参与普通的运算。另外 `m.eval(表达式)` 传入的是表达式，返回的是求值结果，这里是 `x > y` 的真假。

`IntNumRef` 也有 `decl()`，但它返回的不是变量的声明，而是 `Int` 这种值的种类，与 `x.decl()`(变量 `x` 的声明)完全是两回事，注意不要混用。

In [12]:
print(f'{m[x].decl().name() = }')
print(f'{x.decl().name() = }')
print(m[x].decl().eq(x.decl()))

m[x].decl().name() = 'Int'
x.decl().name() = 'x'
False


`m[x].decl()` 的名字是 `'Int'`，`x.decl()` 的名字是 `'x'`，调用的明明是同一个 `decl()`，可对着不同的对象调用，含义就变了。

## 核对继承关系

`type(obj).mro()` 能够把一个类的祖先一路列出来，据此可以核对前面那张类图与实际是否一致。

In [13]:
print(type(x).mro())
print(type(x.decl()).mro())
print(type(m).mro())

[<class 'z3.z3.ArithRef'>, <class 'z3.z3.ExprRef'>, <class 'z3.z3.AstRef'>, <class 'z3.z3.Z3PPObject'>, <class 'object'>]
[<class 'z3.z3.FuncDeclRef'>, <class 'z3.z3.AstRef'>, <class 'z3.z3.Z3PPObject'>, <class 'object'>]
[<class 'z3.z3.ModelRef'>, <class 'z3.z3.Z3PPObject'>, <class 'object'>]


`ArithRef` 一路往上依次是 `ExprRef`、`AstRef`、`Z3PPObject`，而 `FuncDeclRef` 直接挂在 `AstRef` 下面，与前面那张类图对得上。

下面再把 `AstRef` 的子类实际列一遍，看看其中有多少个是与类型相关的类。

In [14]:
print([cls.__name__ for cls in z3.AstRef.__subclasses__()])


def print_tree(cls, indent=0, depth=2):
    print("  " * indent + cls.__name__)
    if depth == 0:
        return
    for sub in cls.__subclasses__():
        print_tree(sub, indent + 1, depth - 1)


print()
print_tree(z3.SortRef, depth=1)

['SortRef', 'FuncDeclRef', 'ExprRef']

SortRef
  TypeVarRef
  BoolSortRef
  ArithSortRef
  BitVecSortRef
  ArraySortRef
  FiniteSetSortRef
  DatatypeSortRef
  FiniteDomainSortRef
  FPSortRef
  FPRMSortRef
  SeqSortRef
  CharSortRef
  ReSortRef


`SortRef` 之下排着一串按类型划分的类，例如 `BoolSortRef`、`ArithSortRef`、`BitVecSortRef` 等等，前面几章用过的 `BitVecSort`、`RealSort`、`IntSort` 也全都属于 `SortRef` 这一支。

## 汇总成一张表

| 想知道什么 | 用哪个 | 返回的类型 |
|---|---|---|
| 变量或表达式的类型 | `expr.sort()` | `SortRef` |
| 表达式子节点的个数 | `expr.num_args()` | `int` |
| 表达式的第 i 个子节点 | `expr.arg(i)` | `ExprRef` |
| 函数声明 | `expr.decl()` | `FuncDeclRef` |
| 函数的名称、参数、类型 | `name()`, `arity()`, `range()`, `domain(i)` | 字符串、数、`SortRef` |
| 模型的值 | `m[x]` | `IntNumRef` |
| 转成 Python 整数 | `.as_long()` | `int` |

表中所列的方法覆盖了从变量到类型、从声明到取值的全部查询路径。熟悉这张表以后，再遇到陌生的 Z3 对象，就知道该调用哪个方法去查看。

- `Int`、`Bool`、`x > y`，本质上都是 `ExprRef`(表达式)
- 类型归 `SortRef`，函数声明归 `FuncDeclRef`，表达式归 `ExprRef`，三者都在 `AstRef` 门下
- `model[0]` 给出的是声明，`model[x]` 给出的是值，同一个 `[]` 有两种含义